# 01 — Exploração dos dados (EDA)

Esse notebook aborda:

> TODO (Gabriel): listar as etapas do checklist cobertas aqui.

## Roteiro e fontes

Cada seção abre com as **perguntas** que ela responde e a **fonte** de onde a pergunta vem. Cada resposta vira uma linha de **Insights**, e cada insight vira uma **decisão** para a etapa 4 (preparar os dados), registrada na seção 3.9.

| Sigla | Fonte | Onde ler |
|---|---|---|
| **Géron** | Aurélien Géron, *Hands-On Machine Learning*, apêndice "Checklist de um projeto de Machine Learning" (tradução Asimov) | `o-checklist-de-um-projeto-de-machine-learning.pdf`, no zip *Fundamentos de AI e Machine Learning* da Asimov |
| **sklearn** | scikit-learn, *Common pitfalls and recommended practices* (seções 12.2 e 12.3) | https://scikit-learn.org/stable/common_pitfalls.html |
| **MLCC-num** | Google ML Crash Course, *Numerical data: First steps* | https://developers.google.com/machine-learning/crash-course/numerical-data/first-steps |
| **MLCC-qual** | Google ML Crash Course, *Datasets: Data characteristics* | https://developers.google.com/machine-learning/crash-course/overfitting/data-characteristics |
| **MLCC-desb** | Google ML Crash Course, *Datasets: Class-imbalanced datasets* | https://developers.google.com/machine-learning/crash-course/overfitting/imbalanced-datasets |
| **MLCC-div** | Google ML Crash Course, *Datasets: Dividing the original dataset* | https://developers.google.com/machine-learning/crash-course/overfitting/dividing-datasets |

Regra que atravessa o notebook: **o conjunto de teste é separado antes da exploração e não é olhado depois** (Géron 2.11; sklearn 12.2: *"Always split the data into train and test subsets first"*).

## Setup

In [89]:
from pathlib import Path
from IPython.display import display, Markdown
from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


SEED = 42
DATA_DIR = Path("../data")

## 1. Objetivo

**Perguntas** (Géron, etapa 1):
- Qual é o objetivo do ponto de vista do negócio, e como a solução seria usada? (1.1, 1.2)
- Como enquadrar o problema: supervisionado ou não, classificação ou regressão, online ou offline? (1.4)
- Como medir o desempenho, e essa medida está alinhada ao objetivo? (1.5, 1.6)
- Qual é o desempenho mínimo aceitável? Qual é a referência de um "chute"? (1.7)
- Como você resolveria o problema manualmente? (1.10)

> TODO (Gabriel).

## 2. Obter e carregar os dados

In [90]:
df_cardio = pd.read_csv(DATA_DIR / "cardio_train.csv", index_col=0)

### 2.1 Tamanho, tipo e duplicatas

**Perguntas:**
- Quantas linhas e colunas existem, e de que tipo é o conjunto (amostra de pacientes, série temporal...)? (Géron 2.10)
- Há linhas repetidas? Repita a checagem **sem as colunas que são só identificadores**, porque uma coluna única por linha esconde as duplicatas. (MLCC-qual: *duplicate examples* entre as causas de dados não confiáveis)
- O que fazer com as duplicatas? (MLCC-div: remover da validação e do teste os exemplos duplicados do treino)

In [91]:
display(df_cardio.shape)
display(df_cardio.describe().round(2))
display(df_cardio.head())

(70000, 13)

,id,age,gender,height,weight,ap_hi,ap_lo,cholesterol,gluc,smoke,alco,active,cardio
count,70000.00,70000.00,70000.00,70000.00,70000.00,70000.00,70000.00,70000.00,70000.00,70000.00,70000.00,70000.0,70000.0
mean,49972.42,19468.87,1.35,164.36,74.21,128.82,96.63,1.37,1.23,0.09,0.05,0.8,0.5
std,28851.30,2467.25,0.48,8.21,14.40,154.01,188.47,0.68,0.57,0.28,0.23,0.4,0.5
min,0.00,10798.00,1.00,55.00,10.00,-150.00,-70.00,1.00,1.00,0.00,0.00,0.0,0.0
25%,25006.75,17664.00,1.00,159.00,65.00,120.00,80.00,1.00,1.00,0.00,0.00,1.0,0.0
50%,50001.50,19703.00,1.00,165.00,72.00,120.00,80.00,1.00,1.00,0.00,0.00,1.0,0.0
75%,74889.25,21327.00,2.00,170.00,82.00,140.00,90.00,2.00,1.00,0.00,0.00,1.0,1.0
max,99999.00,23713.00,2.00,250.00,200.00,16020.00,11000.00,3.00,3.00,1.00,1.00,1.0,1.0


,id,age,gender,height,weight,ap_hi,ap_lo,cholesterol,gluc,smoke,alco,active,cardio
0,0,18393,2,168,62.0,110,80,1,1,0,0,1,0
1,1,20228,1,156,85.0,140,90,3,1,0,0,1,1
2,2,18857,1,165,64.0,130,70,3,1,0,0,0,1
3,3,17623,2,169,82.0,150,100,1,1,0,0,1,1
4,4,17474,1,156,56.0,100,60,1,1,0,0,0,0


In [92]:
display(df_cardio.duplicated().sum())
display(df_cardio.drop(columns="id").duplicated().sum())

np.int64(0)

np.int64(24)

In [93]:
display(df_cardio.loc[df_cardio.drop(columns="id").duplicated()])

,id,age,gender,height,weight,ap_hi,ap_lo,cholesterol,gluc,smoke,alco,active,cardio
10562,15094,20495,1,165,70.0,120,80,1,1,0,0,1,0
21784,31110,16793,1,165,68.0,120,80,1,1,0,0,1,0
38505,54977,18988,1,164,65.0,120,80,1,1,0,0,1,0
40365,57690,14552,1,158,64.0,120,80,1,1,0,0,1,0
42450,60642,18353,1,169,67.0,120,80,1,1,0,0,1,0
44653,63776,16937,2,170,70.0,120,80,1,1,0,0,0,0
45125,64445,21280,1,165,65.0,120,80,1,1,0,0,1,0
45748,65352,22077,1,175,69.0,120,80,1,1,0,0,1,1
45810,65438,21230,1,164,62.0,120,80,1,1,0,0,1,0
48917,69842,21945,1,165,60.0,120,80,1,1,0,0,1,0


#### Insights

- Estrutura de 70.000 linhas por 13 colunas.
- Dataset parece ser uma combinação de colunas numéricas e colunas categóricas.
- Idade possui valores altoss de mais para serem tratados em anos. Valor provavelmente em dias
- "cardio" é a coluna Y da pipeline
- O dataset possui 24 valores perfeitamente duplicados. 
  - São 24 pares (48 linhas). As duas linhas de cada par têm a **idade em dias idêntica**. Como o dataset tem cerca de 7,8 mil idades diferentes, é improvável que seja coincidência: o mais provável é o mesmo registro duplicado.
  - Remover as duplicatas **antes** da divisão, para que uma cópia não caia no treino e a outra no teste.

### 2.2 Separar o conjunto de teste

**Perguntas:**
- Que fração vai para o teste, e por que separar **antes** de explorar? (Géron 2.11: *"coloque-o de lado e nunca olhe para ele"*; sklearn 12.2)
- O teste precisa manter a proporção do alvo? (`stratify`)
- Por que fixar `random_state`? (sklearn 12.3: resultados reproduzíveis)

In [94]:
Y = df_cardio["cardio"]
X = df_cardio.drop(columns="cardio")

x_train, x_test, y_train, y_test = train_test_split(X, Y, test_size=0.3, stratify=Y, random_state=SEED)

## 3. Análise exploratória dos dados

A partir daqui, **só o treino** (`x_train`, `y_train`).

### 3.1 Cada atributo: tipo, completude e valores únicos

**Perguntas** (Géron 3.3):
- Qual é o **tipo real** de cada coluna (contínua, inteira, categórica ordinal, binária)? O tipo do pandas diz a verdade?
- Qual é a porcentagem de valores ausentes?
- Que colunas não têm **utilidade** para a tarefa (só identificam a linha)?

In [95]:
display(
    pd.DataFrame({
        "Coluna": x_train.columns,
        "Tipo": x_train.dtypes.values,
        "Completude": ((x_train.notnull().sum())/x_train.shape[0])*100,
        "Valores Únicos": x_train.nunique()
    })
)
#x_train.hist(bins=50,figsize=(15,10))


,Coluna,Tipo,Completude,Valores Únicos
id,id,int64,100.0,49000
age,age,int64,100.0,7798
gender,gender,int64,100.0,2
height,height,int64,100.0,100
weight,weight,float64,100.0,244
ap_hi,ap_hi,int64,100.0,143
ap_lo,ap_lo,int64,100.0,141
cholesterol,cholesterol,int64,100.0,3
gluc,gluc,int64,100.0,3
smoke,smoke,int64,100.0,2


#### Insights

- Colunas categóricas = ["gender", "cholesterol", "gluc", "smoke", "alco", "active"]. Entre eles, "gender", "smoke", "alco" e "active" assumem valores binários. Enquanto "gluc" e "cholesterol" assumem valores ordinais(1,2 e 3).
- Colunas numéricas = ["age", "height", "weight", "ap_hi", "ap_lo"]
- Não há necessidade atual de alterar a tipagem dos dados.
- Dados completamente preenchidos
- Coluna id não tem utilidade para a pipeline.


In [96]:
x_train = x_train.drop(columns="id")
x_test = x_test.drop(columns="id")

### 3.2 Distribuição dos atributos numéricos

**Perguntas:**
- Como cada atributo numérico se distribui? Há valores muito afastados do resto? (Géron 3.3 "ruído" e "tipo de distribuição"; MLCC-num: *"Graphs can help you find anomalies or patterns hiding in the data"*)
- A distância entre o percentil 0 e o 25 é muito diferente da distância entre o 75 e o 100? Esse é o sinal de outliers do MLCC-num.

In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

fig = make_subplots(rows=5, cols=1)

fig.add_trace(go.Box(x=x_train["age"]/365, name="Idade"), row=1, col=1)
fig.add_trace(go.Box(x=x_train["weight"], name="peso"), row=2, col=1)
fig.add_trace(go.Box(x=x_train["height"], name="altura"), row=3, col=1)
fig.add_trace(go.Box(x=x_train["ap_hi"], name="pressão_max"), row=4, col=1)
fig.add_trace(go.Box(x=x_train["ap_lo"], name="pressão_min"), row=5, col=1)


fig.update_layout(height=700)

#### Insights

- Coluna Idade: possui poucos outliers e a concentração de registros está em volta do valor 54 anos (já convertidos). A distribuição do gráfico em relação aos valores 0-25 e 75-100 é bem equilibrada, tendo apenas 2 outliers que fogem esse padrão, no espaço fora da lower fence.
- Coluna Peso: possui uma grande quantidade de ouliers, concentrados principalmente apos a Upper Fence. Portanto, há um seério desequilíbrio entre os 0-25 e 75-100 é acentuado.
- Coluna Altura: possui muitos  outliers, tanto abaixo da lower fence (142,5 cm), quanto  acima da upper fence (186,5 cm). Diferente do peso, que concentra acima e só poucos abaixo. O que chama atenção na altura é o mínimo de **59 cm**, impossível para um adulto. (Não há crianças nos dados, conforme o valor mínimo de age)
- Coluna pressão_max: a caixa e os bigodes **foram** gerados, mas ficaram finos demais para aparecer. O eixo vai de −150 a 16.020 por causa dos valores errados, enquanto a caixa (Q1 = 120, Q3 = 140) ocupa só 20 unidades, cerca de 0,1% da largura. Os bigodes (90 a 170) também ficam espremidos num ponto. Conclusão: **outliers extremos destroem a escala do boxplot**; é preciso tratá-los antes de ler a forma da distribuição.
- Coluna pressão_min: mesmo motivo, eixo de −70 a 11.000 e caixa de 80 a 90. .

### 3.3 Frequência dos atributos categóricos

**Perguntas:**
- Quais categorias existem em cada atributo e com que frequência? Há categorias raras? (Géron 3.3)
- O que cada código significa (ex.: `gender` 1 e 2)? Sem documentação, como verificar com os próprios dados?

In [98]:
colunas = ["gender", "cholesterol", "gluc", "smoke", "alco", "active"]
contagem = pd.concat({col: x_train[col].value_counts(dropna=False) for col in colunas})
display(pd.DataFrame({
    "Contagem": contagem,
    "Frequencia (%)": (contagem / len(x_train) * 100).round(2)
}))

Contagem  Frequencia (%)
gender      1     31885           65.07
            2     17115           34.93
cholesterol 1     36656           74.81
            2      6727           13.73
            3      5617           11.46
gluc        1     41605           84.91
            3      3750            7.65
            2      3645            7.44
smoke       0     44694           91.21
            1      4306            8.79
alco        0     46373           94.64
            1      2627            5.36
active      1     39433           80.48
            0      9567           19.52

In [99]:
display(x_train.drop(columns=["gender","gluc", "smoke", "alco", "active"]).groupby("cholesterol").mean().round(2))
display(x_train.drop(columns=["cholesterol","gluc", "smoke", "alco", "active"]).groupby("gender").mean().round(2))
display(x_train.drop(columns=["cholesterol","gender", "smoke", "alco", "active"]).groupby("gluc").mean().round(2))
display(x_train.drop(columns=["cholesterol","gender", "gluc", "alco", "active"]).groupby("smoke").mean().round(2))
display(x_train.drop(columns=["cholesterol","gender", "gluc", "smoke", "active"]).groupby("alco").mean().round(2))
display(x_train.drop(columns=["cholesterol","gender", "gluc", "smoke", "alco"]).groupby("active").mean().round(2))

,age,height,weight,ap_hi,ap_lo
cholesterol,,,,,
1,19278.67,164.58,73.11,127.33,93.87
2,19602.48,163.92,76.63,136.39,107.80
3,20522.96,163.42,78.85,134.11,104.64


,age,height,weight,ap_hi,ap_lo
gender,,,,,
1,19507.59,161.36,72.64,128.58,95.35
2,19387.84,169.94,77.26,130.81,100.11


,age,height,weight,ap_hi,ap_lo
gluc,,,,,
1,19374.40,164.42,73.52,128.76,95.71
2,19686.19,164.10,79.18,135.06,107.94
3,20265.12,163.90,77.51,130.39,100.89


,age,height,weight,ap_hi,ap_lo
smoke,,,,,
0,19501.11,163.87,73.94,129.47,96.74
1,19098.85,169.36,77.50,128.18,99.88


,age,height,weight,ap_hi,ap_lo
alco,,,,,
0,19483.07,164.18,74.03,129.32,96.33
1,19160.15,167.50,78.17,129.95,109.18


,age,height,weight,ap_hi,ap_lo
active,,,,,
0,19503.72,164.53,74.72,129.73,95.52
1,19456.55,164.31,74.14,129.26,97.38


### Insights

- Considerando a relação entre "gender" e "height", pode-se afirmar que o valor "1" refere-se a feminino, enquanto o valor "2" refere-se a masculino.
- Portanto, no dataset há uma distribuiçao maior de mulheres do que homens.
- Há categorias raras, como: 
  - Níveis 2 e 3 "cholesterol" e "gluc".
  - Categoria 1 de "smoke", "alco".
  - Em active a categoria rara é a 0 (19,5%). 
  
Considerando o padrão adotado 0 (ausência) 1 (presença):
    - Valor 0: "não fuma" e "não bebe"
    - Valor 1: "fuma" e "bebe"

Sem uma relação com o valor alvo, não é possível inferir mais classes.

In [100]:

fig = make_subplots(rows=2, cols=3)
fig.add_trace(go.Bar(x=["Feminino", "Masculino"], y=x_train["gender"].value_counts().sort_index(), name="Genero"), row=1, col=1)
fig.add_trace(go.Bar(x=["Gravidade 1", "Gravidade 2", "Gravidade 3"], y=x_train["cholesterol"].value_counts().sort_index(), name="Nível Colesterol"),row=1, col=2)
fig.add_trace(go.Bar(x=["Gravidade 1", "Gravidade 2", "Gravidade 3"], y=x_train["gluc"].value_counts().sort_index(), name="Nível Gluc"),row=1, col=3)
fig.add_trace(go.Bar(x=["Não Fuma", "Fumante"], y=x_train["smoke"].value_counts().sort_index(), name="Fuma"), row=2, col=1)
fig.add_trace(go.Bar(x=["Não Bebe", "Alcolatra"], y=x_train["alco"].value_counts().sort_index(), name="Bebe?"), row=2, col=2)
fig.add_trace(go.Bar(x=["not Active", "Active"], y=x_train["active"].value_counts().sort_index(), name="Se exercita?"), row=2, col=3)

### 3.4 Cópia para exploração e atributos derivados

**Perguntas:**
- Por que explorar numa **cópia** e não nos dados originais? (Géron 3.1 e nota da etapa 4: *"mantenha o conjunto de dados original intacto"*)
- Que atributos derivados ajudam a raciocinar (unidade mais natural, combinação de colunas)? Eles aqui são **só para explorar**: as transformações de verdade entram no notebook 02, dentro de um `Pipeline` (Géron 3.8; sklearn 12.2.2).

In [101]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

df_explorar = x_train.join(y_train)
df_explorar["idade_anos"] = df_explorar["age"] / 365.25
df_explorar["imc"] = df_explorar["weight"] / (df_explorar["height"] / 100) ** 2

colunas_numericas = ["idade_anos", "height", "weight", "ap_hi", "ap_lo", "imc"]
colunas_categoricas = ["gender", "cholesterol", "gluc", "smoke", "alco", "active"]

### 3.5 Unidades e valores fisicamente possíveis

**Perguntas:**
- Em que unidade está cada atributo? Os valores mínimo, máximo e os percentis extremos são **possíveis para um ser humano**? (Géron 3.3; MLCC-num: média, mediana, desvio padrão e percentis)
- A distribuição é simétrica ou assimétrica (`skew`)? Que transformação isso sugere? (Géron 3.3 "tipo de distribuição"; Géron 4.3 log, raiz)
- MLCC-num alerta: *"Don't over-rely on basic statistics"*. O que o gráfico mostra que a tabela esconde?

In [102]:
display(df_explorar[colunas_numericas].describe(percentiles=[0.001, 0.01, 0.25, 0.5, 0.75, 0.99, 0.999]).round(1))
display(df_explorar[colunas_numericas].skew().round(2))

,idade_anos,height,weight,ap_hi,ap_lo,imc
count,49000.0,49000.0,49000.0,49000.0,49000.0,49000.0
mean,53.3,164.4,74.3,129.4,97.0,27.6
std,6.7,8.2,14.4,172.5,194.8,6.2
min,29.7,59.0,21.0,-150.0,-70.0,8.0
0.1%,39.2,120.0,40.0,12.0,40.0,16.0
1%,39.6,147.0,48.0,90.0,60.0,18.6
25%,48.4,159.0,65.0,120.0,80.0,23.9
50%,53.9,165.0,72.0,120.0,80.0,26.4
75%,58.4,170.0,82.0,140.0,90.0,30.2
99%,64.3,184.0,118.0,180.0,1000.0,44.6


idade_anos    -0.31
height        -0.71
weight         1.02
ap_hi         76.07
ap_lo         32.02
imc            8.07
dtype: float64

#### Insights


- **Idade:** de 29,7 a 64,9 anos, plausível.
- **Altura:** mediana de 165 cm, mas mínimo de 59 e máximo de 250. Até o percentil 0,1% (120 cm) os valores são possíveis;.
- **Peso:** de 21 a 200 kg. Abaixo de 30 kg há uma linha só e deve ser considerado irreal. 200 kg é raro, mas possível.
- **`ap_hi`:** entre os percentis 1% e 99% vai de 90 a 180 mmHg, o que é plausível. Mas o mínimo é −150 e o máximo é 16.020, o que é impossível. 
- **`ap_lo`:** o percentil 99% já é **1.000**, ou seja, mais de 1% do treino (656 linhas) tem `ap_lo` ≥ 1.000. Parece um zero a mais (100 → 1000).
- **IMC:** máximo de 298,7, impossível. O IMC herda os erros de altura e peso (50 linhas com IMC > 60).
- **Assimetria (`skew`):** os valores de 76 (`ap_hi`) e 32 (`ap_lo`) não descrevem a forma real da pressão, e sim os erros. 
- **O que a tabela esconde (MLCC-num, *"Don't over-rely on basic statistics"*):** a média de `ap_lo` (97) parece normal, mas está puxada pelos 1.000s. Só os percentis extremos e o gráfico revelam o problema.

### 3.6 Ruído e outliers: erro ou caso real?

**Perguntas:**
- Quais valores violam limites físicos ou regras entre colunas (ex.: pressão diastólica maior que a sistólica)? (MLCC-qual: *bad feature values*; verificação automática por intervalo definido)
- Os valores absurdos seguem um **padrão** (ex.: erro de digitação)? Isso decide entre **corrigir** e **apagar**.
- O MLCC-num separa: outlier **por erro** → apagar; outlier **legítimo** → manter ou recortar (*clipping*).
- Quantas linhas cada regra remove? A taxa do alvo nas linhas removidas é diferente da taxa nas que ficam? Remover cria **viés**?
- Os limites usados são defensáveis? O Géron recomenda, na etapa 3, *"tente obter informações de um especialista de campo"*.

In [113]:
regras = {
    "pressão negativa ou zero": (df_explorar["ap_hi"] <= 0) | (df_explorar["ap_lo"] <= 0),
    "ap_hi fora de 60–250": ~df_explorar["ap_hi"].between(60, 250),
    "ap_lo fora de 30–200": ~df_explorar["ap_lo"].between(30, 200),
    "ap_lo maior que ap_hi": df_explorar["ap_lo"] > df_explorar["ap_hi"],
    "altura fora de 120–220 cm": ~df_explorar["height"].between(120, 220),
    "peso fora de 30–200 kg": ~df_explorar["weight"].between(30, 200),
}

display(pd.DataFrame({
    "Linhas": {nome: mascara.sum() for nome, mascara in regras.items()},
    "% do treino": {nome: round(mascara.mean() * 100, 2) for nome, mascara in regras.items()},
}))

outliers_irreais = pd.concat(regras, axis=1).any(axis=1)
display(f"Linhas que alguma regra descartaria: {outliers_irreais.sum()} ({outliers_irreais.mean():.2%})")

,Linhas,% do treino
pressão negativa ou zero,18,0.04
ap_hi fora de 60–250,158,0.32
ap_lo fora de 30–200,714,1.46
ap_lo maior que ap_hi,869,1.77
altura fora de 120–220 cm,44,0.09
peso fora de 30–200 kg,1,0.00


'Linhas que alguma regra descartaria: 979 (2.00%)'

In [104]:
display(df_explorar.loc[regras["ap_hi fora de 60–250"], ["ap_hi", "ap_lo"]].value_counts().head(15))
display(df_explorar.loc[regras["ap_lo fora de 30–200"], ["ap_hi", "ap_lo"]].value_counts().head(15))
display(df_explorar.loc[regras["ap_lo maior que ap_hi"], ["ap_hi", "ap_lo"]].value_counts().head(15))

## Acho melhor reestruturar essa parte do código para a que haja uma representação melhor das informações, assim como está ficou confuso.

ap_hi  ap_lo
12     80       45
14     90       12
11     70        9
14     80        4
13     90        4
14     100       3
13     80        3
906    0         3
11     60        3
10     70        3
17     80        2
907    70        2
1400   90        2
12     90        2
1420   80        2
Name: count, dtype: int64

ap_hi  ap_lo
150    1000     146
160    1000     125
140    1000     100
180    1000      32
160    1100      32
170    1000      27
150    1100      25
130    1000      20
180    1100      16
140    1100      12
170    1100      12
190    1000       8
120    1000       6
160    1200       6
200    1000       4
Name: count, dtype: int64

ap_hi  ap_lo
150    1000     146
160    1000     125
140    1000     100
12     80        45
180    1000      32
160    1100      32
170    1000      27
150    1100      25
130    1000      20
80     120       17
180    1100      16
140    1100      12
170    1100      12
14     90        12
11     70         9
Name: count, dtype: int64

In [105]:
display(df_explorar.groupby(outliers_irreais)["cardio"].agg(["count", "mean"]).rename(index={False: "fica", True: "sai"}))

,count,mean
fica,48021,0.494679
sai,979,0.745659


In [106]:
df_limpo = df_explorar[~outliers_irreais]

fig = make_subplots(rows=len(colunas_numericas), cols=1)
for i, col in enumerate(colunas_numericas):
    fig.add_trace(go.Box(x=df_limpo[col], name=col), row=i + 1, col=1)
fig.update_layout(height=900, showlegend=False)

#### Insights

- **Quanto sai:** 979 linhas (2,0% do treino) violam pelo menos uma regra. Quase tudo vem da pressão. As regras "`ap_lo` > `ap_hi`" (869) e "`ap_lo` fora de 30–200" (714) se sobrepõem, porque os 1.000/1.100 caem nas duas.
- **Padrões dos erros** (tabelas de `value_counts`):
  - `ap_hi` entre 11 e 17 com `ap_lo` entre 60 e 100: pressão em cmHg ou um zero esquecido → **corrigível** (×10).
  - `ap_lo` de 1.000, 1.100 ou 1.200: zero a mais → **corrigível** (÷10).
  - 80/120: colunas trocadas → **corrigível** (inverter).
  - 906/0, 1400/90: sem padrão claro → **apagar**.
- **Viés:** as linhas removidas têm 74,6% de `cardio = 1`, contra 49,5% nas que ficam. Remover tira proporcionalmente mais doentes. Com 2% das linhas o efeito é pequeno, mas mostra que **esses erros não são aleatórios**. Por isso vale testar no notebook 02 **corrigir × apagar**, tratando a escolha como hiperparâmetro (Géron 6.1).
- **Pendente:** os limites das regras não têm fonte clínica. O Géron recomenda, na etapa 3, consultar um especialista de campo.

### 3.7 O alvo

**Perguntas:**
- Qual é o atributo alvo, e qual é a proporção de cada classe? (Géron 3.4; MLCC-desb)
- Qual é a acurácia de um "chute" que sempre prevê a classe mais comum? É a referência mínima da etapa 1.
- Num problema de saúde, que erro custa mais caro (falso negativo ou falso positivo), e que métrica mede esse erro? (Géron 1.5–1.6; MLCC-desb: *"Accuracy is usually a poor metric"* quando as classes são desbalanceadas)

In [107]:
display(y_train.value_counts(normalize=True).round(4))
display(f"Acurácia de sempre prever a classe mais comum: {y_train.value_counts(normalize=True).max():.2%}")

cardio
0    0.5003
1    0.4997
Name: proportion, dtype: float64

'Acurácia de sempre prever a classe mais comum: 50.03%'

#### Insights

- Atributo alvo é cardio
- Possui duas classes ("0" e "1")
  - Até então, não é possível inferir com segurança o que indica cada um. Porém, a princípio, 0 será considerado pessoas que não possuem doenças cardiacas e 1 pessoas que possuem.
- Classes bem balanceadas.
- Problema de saúde implica em casos onde erro de falso negativo é mais grave. Portanto, nesse contexto, a métrica recall se destaca em relação a precisão. Porém a precisão não pode ser ignorada
- Evidência para "1 = tem a doença": na 3.8, `cardio = 1` tem pressão mais alta e idade maior, coerente com doença cardiovascular.

 

### 3.8 Relação com o alvo e entre atributos

**Perguntas:**
- Como cada atributo numérico se distribui em quem tem e em quem não tem a doença? (Géron 3.5)
- Qual é a taxa da doença em cada categoria? (Géron 3.5)
- Quais atributos andam juntos entre si (redundância)? Use **Spearman** porque ela capta relações só crescentes ou só decrescentes, mesmo sem ser uma reta, e sofre menos com outliers. (Géron 3.6)
- Que regra simples um médico usaria com esses atributos? (Géron 3.7)

In [108]:
display(df_limpo.groupby("cardio")[colunas_numericas].median().round(1))

fig = make_subplots(rows=2, cols=3, subplot_titles=colunas_numericas)
for i, col in enumerate(colunas_numericas):
    fig.add_trace(go.Box(x=df_limpo["cardio"], y=df_limpo[col], name=col), row=i // 3 + 1, col=i % 3 + 1)
fig.update_layout(height=700, showlegend=False)

,idade_anos,height,weight,ap_hi,ap_lo,imc
cardio,,,,,,
0,52.0,165.0,70.0,120.0,80.0,25.5
1,55.8,164.0,75.0,130.0,80.0,27.5


In [109]:
taxa = pd.concat({col: df_limpo.groupby(col)["cardio"].agg(["count", "mean"]) for col in colunas_categoricas})
taxa.columns = ["Pacientes", "Taxa de doença"]
display(taxa.round(3))

Pacientes  Taxa de doença
gender      1      31281           0.493
            2      16740           0.498
cholesterol 1      36006           0.435
            2       6540           0.597
            3       5475           0.765
gluc        1      40804           0.475
            2       3552           0.592
            3       3665           0.623
smoke       0      43807           0.497
            1       4214           0.467
alco        0      45473           0.496
            1       2548           0.471
active      0       9396           0.529
            1      38625           0.486

In [110]:
correlacao = df_limpo.drop(columns="age").corr(method="spearman").round(2)

fig = go.Figure(go.Heatmap(
    z=correlacao.values, x=correlacao.columns, y=correlacao.columns,
    zmin=-1, zmax=1, colorscale="RdBu", text=correlacao.values, texttemplate="%{text}",
))
fig.update_layout(height=650)

#### Insights

Colunas numéricas que mais influentes:
- ap_hi
- ap_lo
- wight
- imc
- age(anos)

Colunas categóricas que mais influentes:
- cholesterol
- gluc

Colunas que devem ser testadas se mais ajudam do que atrapalham:
- height
- gender
- smoke
- alco
- active

### 3.9 Conclusões da exploração

**Perguntas** (Géron 3.8–3.10):
- Que transformações valem a pena? **Só listar**: elas entram no notebook 02, dentro de um `Pipeline` (sklearn 12.2.2).
- Que dados extras ajudariam e não existem no arquivo? (3.9)
- O que você aprendeu? (3.10)

✏️ **[Claude]** Tabela preenchida a partir dos Insights das seções 2.1 a 3.8. Status: **decidido** (aplicar no notebook 02), **a testar** (comparar as opções na validação cruzada, tratando a escolha como hiperparâmetro, Géron 6.1) ou **pendente** (falta informação).

| # | Achado | Decisão para a etapa 4 | Status | Base |
|---|---|---|---|---|
| 1 | 24 pares de linhas idênticas, com a idade em dias igual (2.1) | `drop_duplicates()` **antes** de dividir treino e teste. No notebook 02, a divisão é refeita depois dessa remoção | decidido | MLCC-div |
| 2 | `id` não tem utilidade (3.1) | Descartar | decidido | Géron 3.3 |
| 3 | `age` está em dias (2.1; dicionário de dados) | Converter para anos | decidido | Dicionário de dados |
| 4 | Pressão com erro de magnitude: `ap_hi` 11–17 (cmHg) e `ap_lo` ≥ 1.000 (zero a mais) (3.5, 3.6) | **Corrigir** primeiro (×10 e ÷10) e só **descartar** o que continuar fora do intervalo. Nos dados de treino, as linhas inválidas caem de 936 para 172 | a testar × apagar tudo | Shi et al. 2021 |
| 5 | `ap_lo` > `ap_hi` (3.6) | Pela definição, a sistólica (`ap_hi`) é a pressão com o coração batendo e a diastólica (`ap_lo`), com ele em repouso; logo `ap_hi` ≥ `ap_lo`. **Inverter** as colunas quando as duas estiverem em faixa plausível (76 linhas no treino); senão, descartar | a testar × apagar | AHA |
| 6 | Limites de pressão de 60–250 e 30–200 mmHg (3.6) | Manter, **declarados** como escolha do projeto. A literatura não tem um padrão único (ver "Embasamento das regras") | decidido | Mousavi et al. 2024; AHA |
| 7 | Altura de 59 cm, peso de 21 kg, IMC de 298 (3.5) | Trocar os limites próprios pelos de adultos de Lin et al. 2022: altura de 121,9 a 213,4 cm, peso de 22,7 a 317,5 kg e **IMC de 15 a 90**. A regra do IMC pega 16 linhas que passam nas regras atuais | proposto | Lin et al. 2022 |
| 8 | As linhas removidas têm 74,6% de doentes, contra 49,5% nas que ficam (3.6) | Preferir corrigir a apagar (itens 4 e 5) e registrar a taxa de `cardio` antes e depois da limpeza | a testar | Géron 6.1 |
| 9 | Outliers do boxplot que são valores reais: hipertensos, pressões baixas (3.6) | Manter. Para modelos lineares, testar *clipping* ou um escalonamento menos sensível a extremos | a testar | MLCC-num |
| 10 | `weight` e `imc` assimétricos à direita (3.5) | Testar `log` | a testar | Géron 4.3 |
| 11 | Redundância: `weight`–`imc` 0,85 e `ap_hi`–`ap_lo` 0,74 (3.8) | Em modelos lineares, testar manter só um de cada par; em árvores, manter os dois | a testar | Géron 3.6, 4.2 |
| 12 | `height`, `gender`, `smoke`, `alco` e `active` quase sem correlação com o alvo (3.8) | Manter no início e testar a remoção comparando os resultados. Spearman só mede relações monotônicas; uma interação, como `gender` × `height`, pode importar | a testar | Géron 4.2 |
| 13 | `cholesterol` e `gluc` são ordinais (3.1; dicionário) | Manter como inteiro ordinal. Para modelos lineares, testar *one-hot* | a testar | Géron 4.3 |
| 14 | Escalas diferentes entre atributos (3.5) | `StandardScaler` **dentro do `Pipeline`**, para modelos lineares e KNN; árvores não precisam | decidido | Géron 4.4; sklearn 12.2 |
| 15 | Faixas clínicas de pressão (3.8) | Testar um atributo derivado com as categorias da AHA: normal, elevada, estágio 1, estágio 2 e grave | a testar | AHA |
| 16 | Classes balanceadas; falso negativo é o erro mais grave (3.7) | Métrica principal: **recall junto com precisão** (F2 ou ROC AUC). Referência mínima: 50,03% de acurácia | decidido | Géron 1.5–1.7; MLCC-desb |
| 17 | O significado de `gender`, `smoke`, `alco`, `active` e `cardio` foi inferido (3.3, 3.7) | Registrar no README como inferência, não como fato documentado | pendente | Dicionário de dados |

#### ✏️ **[Claude]** Embasamento das regras de limpeza

| Regra da 3.6 | O que a literatura diz | Fonte | Verificação |
|---|---|---|---|
| Pressão ≤ 0 | Valores não positivos são removidos | Mousavi et al., *Learning from Two Decades of Blood Pressure Data*, IEEE EMBC 2024, seção III-B — https://arxiv.org/abs/2402.01598 | lido |
| `ap_hi` de 60 a 250 | Mousavi et al. mantêm a sistólica entre 20 e 200 mmHg e a diastólica entre 30 e 300 mmHg, faixas que cobrem 99% dos 75 milhões de registros. É um corte **estatístico**: nos nossos dados de treino, ele cortaria 36 sistólicas reais entre 201 e 250. A AHA classifica acima de 180 como hipertensão grave, então valores acima de 200 são possíveis | Mousavi et al. 2024; AHA — https://www.heart.org/en/health-topics/high-blood-pressure/understanding-blood-pressure-readings | lido |
| `ap_lo` de 30 a 200 | O limite inferior (30) coincide com o de Mousavi et al.; o superior (200) é escolha do projeto | Mousavi et al. 2024 | lido |
| `ap_lo` > `ap_hi` | A AHA define a sistólica como a pressão "when the heart beats" e a diastólica como a pressão "while the heart muscle rests between beats". A regra **é dedução** dessa definição; nenhuma fonte lida a enuncia | AHA | lido (regra inferida) |
| Corrigir antes de apagar | "the wrong values can be 9, which should be corrected into 90, or 16,000, which should be corrected into 160"; o que continua fora do intervalo depois da correção vira ausente ("3000 … unreasonable even after converted as 30 or 300") | Shi et al., *An automated data cleaning method for EHR by incorporating clinical knowledge*, BMC Med Inform Decis Mak 21:267, 2021 — https://pmc.ncbi.nlm.nih.gov/articles/PMC8449435 | lido |
| Altura, peso e IMC de adultos | "heights <121.9 cm (4 ft) or >213.4 cm (7 ft)", "weights <22.7 kg (50 lb.) or >317.5 kg (700 lb.)", "BMI <15 or >90 for adults" | Lin et al., *Cleaning of anthropometric data from PCORnet EHR using automated algorithms*, JAMIA Open 5(4), 2022 — https://academic.oup.com/jamiaopen/article/5/4/ooac089/6794261 | lido |